# Experiment E1: ocean resolution 1/12 deg -> 1/24 deg, ERA5 forcing (Hurricane Ike 2008)

**Changes from E2 (`gom12_ike_E2.001`, ERA5 at 1/12 deg):** ocean grid **1/24 deg (~4.5 km)**, full-resolution GEBCO (15 arc-sec), `DT = 300 s`, 256 ocean cores.
**Same as E2:** **ERA5 wind + pressure** (section 4c), JRA for the other atmosphere fields, GLORYS boundaries, TPXO tides, 5 m minimum depth, dates.

**Question:** with ERA5 winds, does a finer ocean grid raise the coastal surge and improve the SFINCS boundary?
**Compare with:** E2 (and E0). Section 7d plots all finished runs.

**Run E2 first** (`ike_E2_era5.ipynb`): it makes the ERA5 files this notebook reuses.
Case `gom24_ike_E1.001`. Plan: `EXPERIMENT_PLAN.md`.

**Where to run:** Casper JupyterHub, kernel **CrocoDash**, **64 GB** memory. Run top to bottom once; do **not** re-run the `Case(...)` cell.


## 0. Settings — edit this cell only

In [ ]:
import os
from pathlib import Path

# --- event: run this notebook once per event -> one CESM case each ---
EVENT = "ike"           # this notebook is for ike only
EVENTS = {
    # 10+ days spin-up before the storm enters the Gulf
    "katrina": ["2005-08-15 00:00:00", "2005-09-05 00:00:00"],  # landfall 29 Aug 2005 (LA/MS)
    "ike":     ["2008-08-30 00:00:00", "2008-09-20 00:00:00"],  # landfall 13 Sep 2008 (Galveston)
}
DATE_RANGE = EVENTS[EVENT]
CASENAME   = "gom24_ike_E1.001"
GRID_NAME  = "gom24_ike_E1"   # unique per case: CESM registers grid names globally

# --- paths (match your CROCODILEworkspace install) ---
WORKSPACE = Path("/glade/work/faezehmaghso/crocodile2026")
CESMROOT  = WORKSPACE / "CESM"                      # CESM checkout made by install.sh
INPUTDIR  = WORKSPACE / "croc_input" / CASENAME
CASEROOT  = WORKSPACE / "croc_cases" / CASENAME

PROJECT   = "UCGD0009"   # workshop project; replace with your own NCAR project code

# --- data on GLADE (from CrocoGallery/crocogallery/known_paths.json) ---
GEBCO  = Path("/glade/campaign/cgd/oce/projects/CROCODILE/workshops/2026/CrocoDash/data/gebco_2026/GEBCO_2026.nc")   # full resolution (15 arc-sec) for the finer grid
TPXO_H = "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc/crocogallerydata/tpxo/h_tpxo9.v1.nc"
TPXO_U = "/glade/campaign/cesm/cesmdata/inputdata/ocn/mom/croc/crocogallerydata/tpxo/u_tpxo9.v1.nc"

# --- domain: whole Gulf of Mexico + NW Caribbean ---
# Longitudes in [0, 360]:  262 = 98W,  282 = 78W ... east edge at 82W cuts the Florida Straits.
RES    = 1/24      # deg (~4.5 km)  <-- E1
XSTART = 262.0     # 98W
LENX   = 16.0      # -> 82W
YSTART = 18.0
LENY   = 13.0      # -> 31N (north edge on land)
MIN_DEPTH = 5.0    # m
NK = 50            # vertical levels

# --- tides ---
TIDAL_CONSTITUENTS = ["M2", "S2", "N2", "K2", "K1", "O1", "P1", "Q1"]

# --- compute ---
NTASKS_OCN = 256           # more cores for the finer grid
JOB_QUEUE  = "main"
WALLCLOCK  = "02:00:00"

BATHY = GEBCO     # bathymetry file used by the topography cell
assert GEBCO.exists(), f"GEBCO not found: {GEBCO}"
assert CESMROOT.exists(), f"CESM not found: {CESMROOT}"

## 1. Horizontal grid

In [ ]:
from CrocoDash.grid import Grid

grid = Grid(
    resolution=RES,
    xstart=XSTART,
    lenx=LENX,
    ystart=YSTART,
    leny=LENY,
    name=GRID_NAME,
)
print(f"nx={grid.nx}, ny={grid.ny}, cells={grid.nx * grid.ny:,}")

## 2. Topography

GEBCO 2026 coarsened ×8 (~2 arc-min), enough for 1/12°. `keep_largest_basin()` removes lakes and isolated ponds.

In [ ]:
from CrocoDash.topo import Topo

topo = Topo(grid=grid, min_depth=MIN_DEPTH)
topo.set_from_dataset(
    bathymetry_path=BATHY,
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation",
    depth_method="xesmf",
    mask_method="dataset",
)
topo.keep_largest_basin()
topo.basintmask.plot(size=5, aspect=grid.nx / grid.ny)
print(f"max depth = {float(topo.max_depth):.0f} m")

### 2b. Which edges are open?

Expected: **south** (NW Caribbean → Yucatán inflow) and **east** (Florida Straits outflow).
**north** and **west** should be all land. If not, adjust the domain before continuing.

In [ ]:
import numpy as np

wet = np.asarray(topo.tmask) > 0          # 1 = ocean
assert wet.shape == (grid.ny, grid.nx), wet.shape

edges = {"south": wet[0, :], "north": wet[-1, :], "west": wet[:, 0], "east": wet[:, -1]}
for name, e in edges.items():
    print(f"{name:5s}: {int(e.sum()):4d} wet of {e.size}")

BOUNDARIES = [name for name, e in edges.items() if e.any()]
print("open boundaries:", BOUNDARIES)
if {"north", "west"} & set(BOUNDARIES):
    print("WARNING: north/west edge has water -- review the domain before going on.")

## 3. Vertical grid

In [ ]:
from CrocoDash.vgrid import VGrid

vgrid = VGrid.hyperbolic(nk=NK, depth=topo.max_depth, ratio=50.0)
print("top layer thickness ~", float(vgrid.zi[1] - vgrid.zi[0]), "m")

## 4. CESM case + forcings

`CR_JRA` = ocean-only MOM6 with JRA55-do data atmosphere. Sea-level pressure is passed to MOM6, so the inverse-barometer effect is in η (report §A2).

In [ ]:
# Safety check: stop if anything points at another case, or the case already exists
for p in [CASEROOT, INPUTDIR]:
    assert "gom24_ike_E1" in str(p), f"STOP: {p} is not a gom24_ike_E1 folder"
assert not CASEROOT.exists(), f"STOP: {CASEROOT} already exists (do not re-run Case; go to section 7)"
print("Safe: case will be created in")
print("  ", CASEROOT)
print("  ", INPUTDIR)

In [ ]:
from CrocoDash.case import Case

case = Case(
    cesmroot=CESMROOT,
    caseroot=CASEROOT,
    inputdir=INPUTDIR,
    ocn_grid=grid,
    ocn_vgrid=vgrid,
    ocn_topo=topo,
    compset="CR_JRA",
    machine="derecho",
    project=PROJECT,
    ntasks_ocn=NTASKS_OCN,
    job_queue=JOB_QUEUE,
    job_wallclock_time=WALLCLOCK,
    override=True,
)

In [ ]:
# Sets RUN_STARTDATE / STOP_N from DATE_RANGE and aligns JRA stream years to the event year.
case.configure_forcings(
    date_range=DATE_RANGE,
    boundaries=BOUNDARIES,
    product_name="GLORYS",
    function_name="get_glorys_data_from_rda",
    tpxo_elevation_filepath=TPXO_H,
    tpxo_velocity_filepath=TPXO_U,
    tidal_constituents=TIDAL_CONSTITUENTS,
)

In [ ]:
# Slowest step: cuts GLORYS IC + OBCs from RDA, regrids them, and regrids TPXO onto each open edge.
case.process_forcings()

### 4b. Extra MOM6 parameters

- CrocoDash turns on only `TIDE_M2` for the body (astronomical) tide (`CrocoDash/forcing/tides.py:86`). Galveston is diurnal, so turn on the rest.
- CrocoDash sets no `DT` for custom grids. 600 s matches the SeaSloth benchmark runs at ~0.1°.

Entries outside CrocoDash's own `! >>> CrocoDash` blocks are kept and take precedence (`Case.configure_forcings` docstring).

In [ ]:
import re

TAG = "! >>> added by 01_build_gulf_mom6.ipynb"
wanted = {"DT": "300.0"}
wanted.update({f"TIDE_{c}": "True" for c in TIDAL_CONSTITUENTS if c != "M2"})

user_nl = CASEROOT / "user_nl_mom"
text = user_nl.read_text()

# Parameters already set anywhere in user_nl_mom (MOM_interface rejects duplicates).
present = set()
for line in text.splitlines():
    s = line.strip()
    if not s or s.startswith("!"):
        continue
    s = s.replace("#override", "").strip()
    m = re.match(r"(\w+)\s*=", s)
    if m:
        present.add(m.group(1))

missing = {k: v for k, v in wanted.items() if k not in present}
if missing:
    block = [TAG] + [f"{k} = {v}" for k, v in missing.items()] + ["! <<< added by 01_build_gulf_mom6.ipynb", ""]
    user_nl.write_text(text.rstrip() + "\n\n" + "\n".join(block))
print("added:", list(missing) or "nothing (all already present)")
print("skipped (already set):", sorted(set(wanted) & present))


### 4c. ERA5 wind and pressure (same forcing as E2)

All experiments now use **ERA5** wind (u10, v10) and sea-level pressure (msl) instead of JRA55-do; the other atmosphere fields stay JRA.
This cell reuses the ERA5 files and mesh made by **`ike_E2_era5.ipynb` sections 2 and 3**, so run those first.
It adds a block to `user_nl_datm_streams` after CrocoDash's block (the last entry wins in CDEPS).

In [ ]:
import re

FORCE_DIR = WORKSPACE / "era5_forcing" / "ike2008"          # made by ike_E2_era5.ipynb
MESH = FORCE_DIR / "era5_0.25deg_global_ESMFmesh.nc"
out_files = {k: FORCE_DIR / f"era5_{k}_ike2008.nc" for k in ["u10", "v10", "msl"]}
missing = [str(p) for p in [MESH, *out_files.values()] if not p.exists()]
assert not missing, f"Run sections 2 and 3 of ike_E2_era5.ipynb first. Missing: {missing}"

TAG_B, TAG_E = "! >>> ERA5 wind + pressure", "! <<< ERA5 wind + pressure"
nl = CASEROOT / "user_nl_datm_streams"
txt = nl.read_text() if nl.exists() else ""
txt = re.sub(re.escape(TAG_B) + r".*?" + re.escape(TAG_E) + r"\n?", "", txt, flags=re.S)

names = set(re.findall(r"^\s*([A-Za-z0-9_.]+)\s*:", txt, re.M))
if not names:
    xml = CASEROOT / "CaseDocs" / "datm.streams.xml"
    if xml.exists():
        names = set(re.findall(r'name="([^"]+)"', xml.read_text()))
assert names, "No DATM stream names found: run ./preview_namelists in the case folder, then re-run this cell."

lines = [TAG_B]
for key, suffix, datavar in [("u10", "U_10", "u10 Sa_u"), ("v10", "V_10", "v10 Sa_v"), ("msl", "SLP_", "msl Sa_pslv")]:
    cand = sorted(n for n in names if n.endswith(suffix))
    assert len(cand) == 1, f"expected one stream ending in {suffix}, found {cand}"
    s = cand[0]
    lines += [f"{s}:datafiles = {out_files[key]}", f"{s}:meshfile = {MESH}", f"{s}:datavars = {datavar}",
              f"{s}:year_first = 2008", f"{s}:year_last = 2008", f"{s}:year_align = 2008", f"{s}:taxmode = limit"]
lines += [TAG_E, ""]
nl.write_text(txt.rstrip() + "\n\n" + "\n".join(lines))
print("\n".join(lines))

## 5. Output for SFINCS (diag_table)

Default output is daily-mean `zos`. That is the **wrong** variable for surge: it adds p_atm/(ρg) and removes the domain mean (report §3).
This step adds:

| file | field | frequency | region |
|---|---|---|---|
| `sfincs_TX` | `SSH_inst` (instantaneous) | 10 min | Texas coast, 97.5–93°W, 27.5–30.0°N (Galveston SFINCS + gauges) |
| `sfincs_MS` | `SSH_inst` | 10 min | LA/MS/AL coast, 91–87.5°W, 28.5–30.8°N (Katrina gauges) |
| `ssh_hourly` | `SSH_inst` (instantaneous) | 1 h | whole domain (maps, sanity checks) |

`regional_section` longitudes use the grid's **0–360** convention.

**No `./preview_namelists` needed first.** The cell writes a complete `SourceMods/src.mom/diag_table`:
- if `CaseDocs/diag_table` exists, it keeps CESM's default outputs and adds ours;
- if not, it writes a small complete table (static grid, daily surface fields) plus ours.

It is safe to re-run: the SFINCS block is never added twice.

**Then check on Derecho** (in the case folder):
```bash
./preview_namelists
grep -c "sfincs_TX\|sfincs_MS\|ssh_hourly" $(./xmlquery --value RUNDIR)/diag_table   # must print 6
```

In [ ]:
# Writes SourceMods/src.mom/diag_table. MOM_interface copies a resolved diag_table from
# SourceMods straight into the run folder, so ./preview_namelists is NOT needed first.
import shutil

src = CASEROOT / "CaseDocs" / "diag_table"          # CESM defaults (exists after a preview_namelists)
dst = CASEROOT / "SourceMods" / "src.mom" / "diag_table"
MARK = "### SFINCS nesting output"                  # everything from here on is ours


def box(w, e, s, n):
    # FMS regional_section: lon0 lon1 lat0 lat1 z0 z1, lon in 0-360 like the grid
    return f"{w % 360:.2f} {e % 360:.2f} {s:.2f} {n:.2f} -1 -1"


c = CASENAME
sfincs = [
    "",
    f"{MARK} (added by notebook section 5)",
    f'"{c}.mom6.sfincs_TX", 10, "minutes", 1, "days", "time"',
    f'"{c}.mom6.sfincs_MS", 10, "minutes", 1, "days", "time"',
    f'"{c}.mom6.ssh_hourly", 1, "hours", 1, "days", "time"',
    # wind stress + surface currents, Texas box, every 2 h (posted once per 1-h coupling step,
    # so a 2-h instantaneous window always holds data)
    f'"{c}.mom6.wind_TX", 2, "hours", 1, "days", "time"',
    f'"ocean_model", "SSH_inst", "SSH_inst", "{c}.mom6.sfincs_TX", "all", ".false.", "{box(-97.5, -93.0, 27.5, 30.0)}", 2',
    f'"ocean_model", "SSH_inst", "SSH_inst", "{c}.mom6.sfincs_MS", "all", ".false.", "{box(-91.0, -87.5, 28.5, 30.8)}", 2',
    # SSH_inst (posted every DT), not the SSH mean: the SSH mean is posted once per coupling
    # step and an hourly average of it can be empty -> FMS "write EMPTY buffer" FATAL.
    f'"ocean_model", "SSH_inst", "SSH_inst", "{c}.mom6.ssh_hourly", "all", ".false.", "none", 2',
] + [
    f'"ocean_model", "{v}", "{v}", "{c}.mom6.wind_TX", "all", ".false.", "{box(-97.5, -93.0, 27.5, 30.0)}", 2'
    for v in ["taux", "tauy", "SSU", "SSV"]
] + [
    "",
]

if src.exists():
    # CESM default outputs, minus any SFINCS block from an earlier run of this cell
    base = src.read_text().split(MARK)[0].rstrip() + "\n"
    how = "CESM default diag_table (CaseDocs) + SFINCS outputs"
else:
    # No CaseDocs yet: write a small, complete diag_table ourselves
    base = "\n".join(
        [f'"MOM6 diagnostic fields table for CESM case: {c}"', "1 1 1 0 0 0", "",
         f'"{c}.mom6.h.static", -1, "days", 1, "days", "time"',
         f'"{c}.mom6.h.sfc_daily", 1, "days", 1, "days", "time"']
        + [f'"ocean_model", "{v}", "{v}", "{c}.mom6.h.static", "all", "none", "none", 2'
           for v in ["areacello", "geolon", "geolat", "deptho", "wet"]]
        + [f'"ocean_model", "{v}", "{v}", "{c}.mom6.h.sfc_daily", "all", ".true.", "none", 2'
           for v in ["tos", "sos", "SSU", "SSV"]]
    ) + "\n"
    how = "minimal diag_table (no CaseDocs yet) + SFINCS outputs"

# MOM6 declares areacello as the AREA measure for deptho. Some minimal or
# previously generated CaseDocs tables omit it, which is fatal at startup.
if '"areacello"' not in base:
    base += f'"ocean_model", "areacello", "areacello", "{c}.mom6.h.static", "all", "none", "none", 2\n'
    how += " + required areacello AREA measure"

dst.parent.mkdir(parents=True, exist_ok=True)
dst.write_text(base + "\n".join(sfincs))
n = sum(1 for line in dst.read_text().splitlines() if any(k in line for k in ("sfincs_TX", "sfincs_MS", "ssh_hourly")))
print(f"wrote {dst}\n  {how}\n  SFINCS lines: {n} (must be 6)")
print("\n".join(sfincs))

## 6. Build and run (Derecho terminal)

Run one line at a time, in the case folder:
```bash
cd /glade/work/faezehmaghso/crocodile2026/croc_cases/gom24_ike_E1.001
./case.setup --reset                       # CrocoDash changed NTASKS_OCN after setup
./preview_namelists
grep -c "sfincs_TX\|sfincs_MS\|ssh_hourly" $(./xmlquery --value RUNDIR)/diag_table   # must print 6
grep -c "wind_TX" $(./xmlquery --value RUNDIR)/diag_table                            # must print 5
grep -c "era5_" $(./xmlquery --value RUNDIR)/datm.streams.xml                         # must print 6 (ERA5 wind + pressure)
./xmlchange JOB_QUEUE=tutorial --force     # workshop queue
./xmlquery JOB_QUEUE,JOB_WALLCLOCK_TIME    # tutorial for case.run and case.st_archive
qcmd -A UCGD0009 -- ./case.build           # ~10 min; ends with MODEL BUILD HAS FINISHED SUCCESSFULLY
./case.submit
qstat -u faezehmaghso                      # R = running
tail -8 CaseStatus                         # must show case.run success + st_archive success
```



Expected run time: **about 20 to 40 min on 256 cores (about 8x E0's work)** (E0 at 1/12 deg took 7.5 min on 128 cores).

**If `preview_namelists` fails** with `TypeError ... PosixPath`, a setting is listed twice in `user_nl_mom`:
```bash
grep -o '^[A-Z_0-9]* *=' user_nl_mom | tr -d ' =' | sort | uniq -d
```

**If the run fails:**
```bash
grep -n -i -B3 -A6 "FATAL" $(./xmlquery --value RUNDIR)/ocn.log.*
```
- CFL / velocity truncation near the coast: lower `DT` (for example to 150) in `user_nl_mom`, then `./case.submit` again (no rebuild).

Section 7 below starts **after** `CaseStatus` shows success.

### Submit on the correct queue (workshop `tutorial` reservation)

The **model run** and the **archiver** must use the workshop queue `tutorial`.
CrocoDash creates the case with `main`, so change it **after** `./case.setup --reset` and **before** `./case.submit`:
```bash
./xmlchange JOB_QUEUE=tutorial --force        # --force: CESM doesn't know this workshop queue
./xmlquery JOB_QUEUE,JOB_WALLCLOCK_TIME       # case.run and case.st_archive: tutorial, 02:00:00
./case.submit                                 # the qsub line must show "-q tutorial"
qstat -u faezehmaghso                         # Queue column shows S7606483 = the tutorial reservation
```
- The **build** (`qcmd -A UCGD0009 -- ./case.build`) always runs on the small `develop` queue. That is normal.
- **Already submitted on `main` by mistake?** Cancel both jobs with `qdel <run-id> <archive-id>` (IDs from `qstat -u faezehmaghso`), then run the lines above.
- `qstat -u faezehmaghso` briefly showing nothing right after submitting is normal; check again or run `tail -8 CaseStatus`.
- **After the workshop** the reservation ends: switch back with `./xmlchange JOB_QUEUE=main --force` (and expect longer waits).


## 7. After the run: join, save, check

The box outputs (`sfincs_TX`, `sfincs_MS`) are written in **pieces** (`.nc.0088`, ...), one per processor.
They stay in the **run folder on scratch**, because the archiver skips them.
The next cell joins the pieces and saves them to `/glade/work/.../mom6_output/<case>/`.

**Calendar:** MOM6 writes `calendar: gregorian`, but means proleptic Gregorian. Without the fix, dates appear **2 days early**.

Run the **settings cell (0)** first if the kernel was restarted.

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
from pathlib import Path

RUNDIR = Path(f"/glade/derecho/scratch/faezehmaghso/{CASENAME}/run")
OUTDIR = WORKSPACE / "mom6_output" / CASENAME
OUTDIR.mkdir(parents=True, exist_ok=True)


def tidy(ds):
    """Box output uses xh_sub01/yh_sub01; rename to xh/yh."""
    ren = {d: d.split("_sub")[0] for d in ds.dims if "_sub" in d}
    ren.update({c: c.split("_sub")[0] for c in ds.coords if "_sub" in c and c not in ren})
    return ds.rename(ren)


def open_mom6(path):
    raw = xr.open_dataset(path, decode_times=False)
    raw.time.attrs["calendar"] = "proleptic_gregorian"   # MOM6's real calendar
    return tidy(xr.decode_cf(raw))


def join_tiles(box):
    out = OUTDIR / f"{CASENAME}.mom6.{box}.nc"
    if not out.exists():
        files = sorted(RUNDIR.glob(f"{CASENAME}.mom6.{box}.nc.*"))
        ds = tidy(xr.open_mfdataset(files, combine="by_coords", decode_times=False))
        ds.load().to_netcdf(out)
        print(f"{box}: joined {len(files)} pieces")
    ds = open_mom6(out)
    print(f"{box}: {out.name} {out.stat().st_size/1e6:.1f} MB, {dict(ds.sizes)}, "
          f"{str(ds.time.values[0])[:16]} -> {str(ds.time.values[-1])[:16]}")
    return ds


tx = join_tiles("sfincs_TX")
ms = join_tiles("sfincs_MS")
try:
    wtx = join_tiles("wind_TX")          # taux, tauy, SSU, SSV (2-hourly)
except Exception as e:
    wtx = None
    print("wind_TX not available:", e)

hourly = OUTDIR / f"{CASENAME}.mom6.ssh_hourly.nc"
if not hourly.exists():
    shutil.copy(RUNDIR / hourly.name, hourly)
g = open_mom6(hourly)
print("ssh_hourly:", dict(g.sizes))

# Custom history names are not archived by the stock CIME rules; preserve
# the daily surface file in permanent work storage alongside hourly SSH.
daily = OUTDIR / f"{CASENAME}.mom6.h.sfc_daily.nc"
if not daily.exists():
    shutil.copy2(RUNDIR / daily.name, daily)
d = open_mom6(daily)
print(f"sfc_daily: {daily.name} {daily.stat().st_size/1e6:.1f} MB, "
      f"{str(d.time.values[0])[:16]} -> {str(d.time.values[-1])[:16]}")

### 7a. Where the model cells are, and model sea level

Left: mean sea level over the run, the gauge (red x) and the nearest **ocean** cell used (white o).
Gauges sit in land cells at 1/12 degree, so the model cell is a few km offshore.
Right: MOM6 sea level at that cell, in the model's own datum (not NAVD88 or MSL).

In [ ]:
def nearest_wet(ds, lon, lat):
    ssh = ds["SSH_inst"]
    wet = ssh.isel(time=0, drop=True).notnull()
    d2 = (((ds.xh - lon) * np.cos(np.deg2rad(lat)))**2 + (ds.yh - lat)**2)
    d2 = d2.where(wet).transpose("yh", "xh").values
    iy, ix = np.unravel_index(np.nanargmin(d2), d2.shape)
    return ssh.isel(yh=iy, xh=ix), float(np.sqrt(d2[iy, ix])) * 111


gauges = {  # name: (lon 0-360, lat, dataset)
    "Galveston Pier 21 (TX)": (360 - 94.793, 29.31, tx),
    "Galveston Bay Entrance N Jetty (TX)": (360 - 94.725, 29.357, tx),
    "Sabine Pass North (TX)": (360 - 93.87, 29.728, tx),
}
landfall = np.datetime64("2008-09-13T07:00")   # Ike landfall at Galveston about 2008-09-13 07 UTC

fig, axes = plt.subplots(len(gauges), 2, figsize=(14, 3.3 * len(gauges)),
                         gridspec_kw={"width_ratios": [1, 2]})
for row, (name, (lon, lat, ds)) in enumerate(gauges.items()):
    s, km = nearest_wet(ds, lon, lat)
    ds["SSH_inst"].mean("time").plot(ax=axes[row, 0], cmap="viridis", add_colorbar=False)
    axes[row, 0].plot(lon, lat, "rx", ms=10, label="gauge")
    axes[row, 0].plot(float(s.xh), float(s.yh), "wo", mfc="none", ms=10, label="model cell")
    axes[row, 0].set_title(name, fontsize=10)
    axes[row, 0].legend(fontsize=8)
    s.plot(ax=axes[row, 1])
    axes[row, 1].axvline(landfall, color="r", ls="--", lw=1)
    axes[row, 1].set_title(f"model cell {float(s.xh):.2f}E, {float(s.yh):.2f}N ({km:.0f} km from gauge); red = ike landfall",
                           fontsize=10)
    axes[row, 1].set_ylabel("SSH_inst (m, model datum)")
plt.tight_layout()

### 7b. Sea level at the gauges vs NOAA

Uses the model cell nearest each gauge that is ocean (gauges sit in land cells at 1/12 degree).
Both series are anomalies from their calm-period mean, so the datum doesn't matter here.
The printed NOAA station name confirms each ID.

In [ ]:
import requests


def nearest_wet(ds, lon, lat):
    ssh = ds["SSH_inst"]
    wet = ssh.isel(time=0, drop=True).notnull()
    d2 = (((ds.xh - lon) * np.cos(np.deg2rad(lat)))**2 + (ds.yh - lat)**2)
    d2 = d2.where(wet).transpose("yh", "xh").values
    iy, ix = np.unravel_index(np.nanargmin(d2), d2.shape)
    return ssh.isel(yh=iy, xh=ix), float(np.sqrt(d2[iy, ix])) * 111


def noaa_wl(station, begin, end, datum="MSL"):
    r = requests.get("https://api.tidesandcurrents.noaa.gov/api/prod/datagetter", params=dict(
        product="water_level", station=station, begin_date=begin, end_date=end,
        datum=datum, units="metric", time_zone="gmt", format="json",
        application="MOM6_SFINCS_check"), timeout=60).json()
    if "error" in r:
        print(station, "->", r["error"])
        return None, None
    df = pd.DataFrame(r["data"])
    s = pd.Series(pd.to_numeric(df["v"], errors="coerce").values, index=pd.to_datetime(df["t"]))
    return s, r["metadata"]["name"]


# Ike landfall at Galveston about 2008-09-13 07 UTC
landfall = np.datetime64("2008-09-13T07:00")
calm = slice("2008-09-01", "2008-09-07")
begin, end = "20080830", "20080919"
stations = {  # name: (NOAA id, lon 0-360, lat, dataset)
    "Galveston Pier 21 (TX)": ("8771450", 360 - 94.793, 29.31, tx),
    "Galveston Bay Entrance N Jetty (TX)": ("8771341", 360 - 94.725, 29.357, tx),
    "Sabine Pass North (TX)": ("8770570", 360 - 93.87, 29.728, tx),
}

fig, axes = plt.subplots(len(stations), 1, figsize=(13, 3.2 * len(stations)), sharex=True)
for ax, (name, (sid, lon, lat, ds)) in zip(axes, stations.items()):
    mod, km = nearest_wet(ds, lon, lat)
    mod = mod.to_series()
    ax.plot(mod.index, mod - mod[calm].mean(), label=f"MOM6 ({km:.0f} km offshore)")
    obs, noaa_name = noaa_wl(sid, begin, end)
    title = name
    if obs is not None:
        ax.plot(obs.index, obs - obs[calm].mean(), label=f"NOAA {sid} {noaa_name}", alpha=0.8)
        both = pd.concat([mod.resample("1h").mean() - mod[calm].mean(),
                          obs.resample("1h").mean() - obs[calm].mean()], axis=1).dropna()
        rmse = float(np.sqrt(((both.iloc[:, 0] - both.iloc[:, 1])**2).mean()))
        title += f": hourly RMSE {rmse:.2f} m"
    ax.axvline(landfall, color="r", ls="--", lw=1)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel("anomaly (m)")
    ax.legend(fontsize=8)
plt.tight_layout()

### 7d. This experiment vs the base run (E0) and NOAA

Same gauges, same anomaly method (calm-period mean removed). The table is saved to your home folder.
A change is useful if `peak_err_m` and the forerunner move toward NOAA without making the tides (before 11 Sep) worse.

In [ ]:
from IPython.display import display

BASE_TX = WORKSPACE / "mom6_output" / "gom12_ike.001" / "gom12_ike.001.mom6.sfincs_TX.nc"
RUNS = {"E0 JRA 1/12": WORKSPACE / "mom6_output" / "gom12_ike.001" / "gom12_ike.001.mom6.sfincs_TX.nc",
        "E2 ERA5 1/12": WORKSPACE / "mom6_output" / "gom12_ike_E2.001" / "gom12_ike_E2.001.mom6.sfincs_TX.nc",
        CASENAME: OUTDIR / f"{CASENAME}.mom6.sfincs_TX.nc"}
RUNS = {k: v for k, v in RUNS.items() if v.exists()}   # skip runs that aren't done yet
pre = slice(str(landfall - np.timedelta64(24, "h")), str(landfall))   # forerunner window

rows = []
fig, axes = plt.subplots(len(stations), 1, figsize=(13, 3.2 * len(stations)), sharex=True)
for ax, (name, (sid, lon, lat, _)) in zip(axes, stations.items()):
    obs, _n = noaa_wl(sid, begin, end)
    o = (obs - obs[calm].mean()).resample("1h").mean() if obs is not None else None
    if o is not None:
        ax.plot(o.index, o, "k", lw=1.5, label="NOAA")
    for label, path in RUNS.items():
        m = nearest_wet(open_mom6(path), lon, lat)[0].to_series()
        m = (m - m[calm].mean()).resample("1h").mean()
        ax.plot(m.index, m, label=label)
        r = {"station": name, "run": label, "peak_model_m": m.max(), "peak_time": m.idxmax(),
             "forerunner_model_m": m[pre].mean()}
        if o is not None:
            both = pd.concat([m, o], axis=1, keys=["m", "o"]).dropna()
            r.update(rmse_m=np.sqrt(((both.m - both.o) ** 2).mean()), bias_m=(both.m - both.o).mean(),
                     peak_obs_m=o.max(), peak_err_m=m.max() - o.max(), forerunner_obs_m=o[pre].mean())
        rows.append(r)
    ax.axvline(landfall, color="r", ls="--", lw=1)
    ax.set_title(name, fontsize=10)
    ax.set_ylabel("anomaly (m)")
    ax.legend(fontsize=8)
plt.tight_layout()

score = pd.DataFrame(rows).round(3)
display(score)
score.to_csv(Path.home() / f"{CASENAME}_vs_base_scorecard.csv", index=False)

### 7e. Wind stress and surface current offshore of Galveston

The strength of the wind stress during 11 to 13 Sep is what builds the forerunner. Compare it between experiments (E0 has no wind output).

In [ ]:
if wtx is None:
    print("no wind_TX output in this run")
else:
    def at_point(da, lon, lat):
        xd = [d for d in da.dims if d.startswith("x")][0]
        yd = [d for d in da.dims if d.startswith("y")][0]
        return da.sel({xd: lon, yd: lat}, method="nearest")

    lon, lat = 360 - 94.6, 29.0          # open water ~30 km off Galveston
    taux, tauy = at_point(wtx["taux"], lon, lat), at_point(wtx["tauy"], lon, lat)
    ssu, ssv = at_point(wtx["SSU"], lon, lat), at_point(wtx["SSV"], lon, lat)
    fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
    ax[0].plot(taux.time, np.hypot(taux.values, tauy.values), label="|wind stress|")
    ax[0].plot(taux.time, taux, lw=0.8, label="taux (east)")
    ax[0].plot(tauy.time, tauy, lw=0.8, label="tauy (north)")
    ax[0].set_ylabel("stress (Pa)")
    ax[1].plot(ssu.time, np.hypot(ssu.values, ssv.values), label="|surface current|")
    ax[1].set_ylabel("current (m/s)")
    for a in ax:
        a.axvline(landfall, color="r", ls="--", lw=1)
        a.legend(fontsize=8)
    ax[0].set_title(f"{CASENAME}: 94.6 W, 29.0 N", fontsize=10)
    plt.tight_layout()

### 7c. Animation of sea level (whole Gulf, hourly)

Anomaly from the run mean, 2008-09-09 to 2008-09-15.
The GIF is saved in `OUTDIR` and copied to your home folder; download it from the JupyterLab file browser (right-click, then Download).

In [ ]:
from matplotlib import animation
from IPython.display import Image

ssh = g["SSH_inst"]
anom = (ssh - ssh.mean("time")).sel(time=slice("2008-09-09", "2008-09-15")).load()

fig, ax = plt.subplots(figsize=(9, 6))
ax.set_facecolor("0.75")
im = ax.pcolormesh(anom.xh, anom.yh, np.ma.masked_invalid(anom.isel(time=0).values),
                   cmap="RdBu_r", vmin=-0.8, vmax=0.8, shading="nearest")
fig.colorbar(im, ax=ax, label="SSH anomaly from run mean (m)")
ax.plot(360 - 94.793, 29.31, "k^", ms=6)
ax.set_xlabel("longitude (deg E)")
ax.set_ylabel("latitude (deg N)")
title = ax.set_title("")


def update(i):
    im.set_array(np.ma.masked_invalid(anom.isel(time=i).values))
    title.set_text(f"MOM6 Gulf 1/12 deg, ike: sea level anomaly {str(anom.time.values[i])[:16]} UTC")
    return im, title


anim = animation.FuncAnimation(fig, update, frames=anom.sizes["time"], interval=100)
gif = OUTDIR / f"{CASENAME}_ssh_ike.gif"
anim.save(gif, writer=animation.PillowWriter(fps=10), dpi=80)
plt.close(fig)
shutil.copy(gif, Path.home() / gif.name)
print("saved", gif, "and a copy in your home folder")
Image(filename=str(gif))